# Reading structure from a social network, recognizing authors from text

This tutorial brings together two seemingly unrelated but actually both belonging to "relations and text" social science methods: **network inference** (in a social network, edges don't arise randomly; what structural tendencies underlie them?) and **stylometry** (can we attribute an anonymous text to its true author based solely on word-usage habits?). Their common foundation is that observed relations or text contain regularities that can be identified by statistical models.

On the network side, we do two things. First, **ERGM (Exponential Random Graph Model)**: it answers "how strong are reciprocity (if you follow me, I'm more likely to follow you back) and transitive closure (friends of friends tend to become friends) as tendencies in this network?" It benchmarks against R's `ergm`/`statnet`, the world champion in social network analysis. But there's an inescapable reality: full ERGM maximum likelihood requires MCMC over an analytically intractable normalization constant, and Python's ecosystem has never had a first-class implementation. So we use **MPLE (Maximum Pseudolikelihood Estimation)** to fill that gap, and honestly label it as "an approximation to MCMC-MLE"—this point will recur because knowing you're using an approximation and where the approximation fails is itself part of methodological literacy. The second thing is problems of the **SAOM (Stochastic Actor-Oriented Model)** kind: between two time points, how do networks rewire and behavior drift? It benchmarks against R's `RSiena`. Full SAOM relies on simulation-based method-of-moments estimation, and Python is similarly blank, so we only do a **descriptive simplified layer**—Jaccard stability between waves, edge formation/dissolution, cross-lagged behavior proxies—rather than pretending to be a true SIENA estimator.

On the stylometry side, we can calculate everything exactly, without owing any approximations. The classic task is **author attribution**: setting aside topic words, the usage frequencies of "the/and/of/to" and other **function words** leave a stable "fingerprint" unique to each author, almost independent of content. We use **Burrows's Delta** to go through the full pipeline—select the most frequent words (MFW), z-score normalize them by corpus, calculate Delta distance (mean absolute difference in normalized frequencies), hierarchical cluster, then attribute by nearest neighbor. It benchmarks against R's `stylo`.

All data is built-in synthetic corpora with **known answers**: in the network generation process we **deliberately embedded reciprocity and transitive closure**, so the correct model should estimate both coefficients as positive; the stylometry corpus is 3 authors × 3 texts each, each person with a distinct function-word distribution, so documents by the same author should cluster together and attribution accuracy should be very high. Testing "can the method recover known answers" is the most direct way to judge whether an analytical pipeline is trustworthy. The entire pipeline is tied together with `socialverse`, a library designed for social science that registers input and output at each step into a reproducible evidence chain—you'll see it at the end.

In [1]:
import os
import sys

# 让本 worktree 的 socialverse 优先于任何已安装版本被导入,并把工作目录切到本 notebook 所在处,
# 使 fig_*.png 与 notebook 同目录(教学用,可安全删除)。
_NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_PKG_ROOT = os.path.dirname(_NB_DIR)
if _PKG_ROOT not in sys.path:
    sys.path.insert(0, _PKG_ROOT)
os.chdir(_NB_DIR)

import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件
import matplotlib.pyplot as plt

# 本 notebook 有几张图是手绘的(socialverse 没有现成的 ERGM/SAOM 图),标题含中文;
# 这里选一个系统里装了的中文字体,避免图上出现「豆腐块」。这只影响绘图,不改任何分析。
for _f in ("Arial Unicode MS", "STHeiti", "Songti SC", "Hiragino Sans GB", "Heiti SC"):
    if _f in {f.name for f in matplotlib.font_manager.fontManager.ttflist}:
        plt.rcParams["font.sans-serif"] = [_f, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

## Loading a directed social network

We first construct a 25-node directed network. The generation process has three layers: first place baseline edges according to latent homophily, then inject reciprocity (when `i→j` exists, `j→i` becomes more likely), finally inject transitive closure (when `i→j→k` exists, `i→k` becomes more likely). These two tendencies are exactly what ERGM will identify next—because we put them in deliberately, **the correct model should estimate their coefficients as positive**.

The edge list is in long format, one directed edge per row: `source` follows `target`. This is the raw input for ERGM and SAOM that follows.

In [2]:
edges = ds.load_network(n=25, seed=0)   # 返回列:source, target
n_nodes = len(set(edges["source"]) | set(edges["target"]))
print(f"节点数 = {n_nodes} · 有向边数 = {len(edges)} · 列 = {list(edges.columns)}")
edges.head(8)

节点数 = 24 · 有向边数 = 101 · 列 = ['source', 'target']


,source,target
0,0,3
1,0,4
2,0,7
3,0,9
4,0,12
5,0,16
6,0,17
7,0,19


Register the edge list into study state. `socialverse` uses a `StudyState` object throughout the pipeline, like a whiteboard with fixed slots; here we write the edge list into the `sources` slot, and subsequent network functions read from there without requiring repeated parameter passing.

In [3]:
st = sv.StudyState()
st.write("sources", "datasets", edges)
print("已填槽:", st.populated())

已填槽: {'sources': ['datasets']}


## ERGM: Estimating reciprocity and transitive closure

The core idea of ERGM is: for each possible edge in a network, whether it appears or not depends on what **structural statistics** it would change for the whole graph. Full maximum likelihood requires summing over all possible graphs (that analytically intractable normalization constant), and `statnet` approximates it with MCMC. We use **MPLE** instead: treat each ordered node pair `(i,j)` as a Bernoulli trial, the response is "does this edge exist," and the predictor is "how many structural statistics would change by adding this edge"—`edges` (density/intercept), `mutual` (does the reverse edge `j→i` already exist), `transitive` (how many two-hop paths `i→k→j` would it close). This makes "edge ~ change statistics" an ordinary logistic regression, and the regression coefficients are the ERGM parameters. The cost of MPLE is that it's biased on high-order dependence (especially triangles), so we honestly label it as an approximation to MCMC-MLE.

The step below fits three terms: `edges + mutual + transitive`. The coefficients are log-odds contributions: if a term is positive, it means "one more unit of this structural tendency increases the log-odds of the edge existing."

In [4]:
st = sv.tl.ergm(st, terms=["edges", "mutual", "transitive"], seed=0)

ergm = st.models["ergm"]
print("方法:", ergm["method"], "| 后端:", ergm["backend"])
print("近似声明:", ergm["approximation"])
print(f"节点 {ergm['n_nodes']} · 边 {ergm['n_edges']} · dyad 观测 {ergm['n_dyads']}")

方法: MPLE (maximum pseudo-likelihood) | 后端: statsmodels.Logit
近似声明: MPLE ≈ MCMC-MLE;伪似然=逐 dyad change-stats 的 logistic 回归,非 statnet::ergm 的 MCMC-MLE(intractable normalizing constant)
节点 24 · 边 101 · dyad 观测 552


Lay out the three coefficients in a table: point estimate, standard error, z-value for each. The criterion is simple—`mutual` and `transitive` should be significantly positive (the tendencies we embedded are recovered), and `edges` should be negative (reflecting the sparse baseline density).

In [5]:
coef, se, z = ergm["coef"], ergm["se"], ergm["z"]
pd.DataFrame(
    [{"term": t, "coef": round(coef[t], 4), "se": round(se[t], 4), "z": round(z[t], 2)}
     for t in ergm["terms"]]
)

,term,coef,se,z
0,edges,-3.3366,0.2499,-13.35
1,mutual,2.3701,0.3018,7.85
2,transitive,0.9115,0.1203,7.57


All three coefficients meet expectations: `mutual ≈ +2.37`, `transitive ≈ +0.91`, both far from zero with large z-values—MPLE recovered the reciprocity and transitive closure we buried in the generation process; `edges ≈ −3.34` is negative, consistent with sparse baseline. This is the first "known parameters recovered" moment in this chapter.

In [6]:
assert coef["mutual"] > 0 and coef["transitive"] > 0, "MPLE 应复原正的 mutual/transitive 系数"
print(f"mutual = {coef['mutual']:+.3f}  → 互惠倾向被复原")
print(f"transitive = {coef['transitive']:+.3f}  → 传递闭合倾向被复原")

mutual = +2.370  → 互惠倾向被复原
transitive = +0.912  → 传递闭合倾向被复原


## Checking goodness of fit: observed network vs. model expectation

Positive coefficients only say "the direction is right"; we still need to see whether the network **generated by these coefficients** looks like the real one. `diagnostics['gof']` uses the fitted MPLE coefficients to do sequential conditional simulation, repeatedly generates networks, and compares several global statistics of the observed network (edge count, reciprocated pair count, transitive triad count, density) against model expectations. This is an honestly labeled **simplified version** of GoF—not the full MCMC diagnostics of `ergm::gof`, but sufficient to see where MPLE fits well and where it overshoots.

In [7]:
gof = st.diagnostics["gof"]
obs, exp, sd = gof["observed"], gof["model_expected"], gof["model_sd"]
pd.DataFrame(
    [{"statistic": k, "observed": round(obs[k], 3),
      "model_exp": round(exp[k], 3), "model_sd": round(sd[k], 3)} for k in obs]
)

,statistic,observed,model_exp,model_sd
0,edges,101.000,21.050,4.599
1,mutual_pairs,32.000,2.750,1.410
2,transitive_triads,227.000,0.950,1.396
3,density,0.183,0.038,0.008


How to read it: first-order quantities like density, the observed value and model expectation are roughly the same order of magnitude; but high-order triangular terms like `transitive_triads`, the observed (227) is far higher than model expectation (≈1). This huge gap is exactly the **known limitation of MPLE**—pseudolikelihood treats each edge as independent, systematically underestimating strong dependent structures like triangle closure. Putting it on the table rather than pretending GoF is all-green is how you responsibly use approximate methods.

## Plotting an ERGM results figure

The plotting module in `socialverse` has no specialized ERGM plot, so here we draw by hand in two panels: left side is the coefficient forest plot (error bars = ±1.96·SE, dashed line at zero effect), right side is the GoF observed vs. model expectation bar comparison.

In [8]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.2))

terms = ergm["terms"]
vals = [coef[t] for t in terms]
errs = [1.96 * se[t] for t in terms]
ypos = np.arange(len(terms))[::-1]
axL.errorbar(vals, ypos, xerr=errs, fmt="o", color="#2b6cb0", capsize=4)
axL.axvline(0, color="grey", lw=1, ls="--")
axL.set_yticks(ypos); axL.set_yticklabels(terms)
axL.set_xlabel("MPLE 系数 (log-odds, ±1.96·SE)")
axL.set_title("ERGM 系数:互惠 / 传递闭合 > 0")

keys = list(obs.keys())
x = np.arange(len(keys)); w = 0.38
axR.bar(x - w / 2, [obs[k] for k in keys], w, label="观测", color="#2b6cb0")
axR.bar(x + w / 2, [exp[k] for k in keys], w, label="模型期望", color="#ed8936")
axR.set_xticks(x); axR.set_xticklabels(keys, rotation=30, ha="right", fontsize=8)
axR.set_title("拟合优度:观测 vs 模型期望"); axR.legend()

fig.tight_layout()
fig.savefig("fig_ergm.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("已保存 fig_ergm.png")

已保存 fig_ergm.png


![ERGM coefficients and goodness of fit](fig_ergm.png)

## SAOM: How networks and behavior co-evolve between two time points

ERGM sees a static network. `RSiena` cares about **dynamics**: from wave1 to wave2, the same group of people—how do edges form and dissolve? And network and individual behavior often influence each other—is it "people around you influence your behavior" (influence) or "you choose whom to connect to based on your behavior" (selection)? Full SAOM uses simulation-based method-of-moments estimation to recover rate/selection/influence parameters; Python has no such implementation. We don't pretend to it, only do a **descriptive diagnostic layer**: Jaccard stability between waves (the very first data-quality indicator SIENA users look at; too low means waves are too far apart and no evolutionary model can be trusted), edge formation/dissolution/persistence counts and rates, Hamming distance, and when given behavior vectors, two cross-lagged proxy measures.

For demonstration, we take wave1 as the registered edge list, resample with a different seed to get wave2 (simulating network rewiring over time), assign each node a behavior score related to in-degree, and let it drift slightly in the wave2 in-degree direction—this way we embed a signal "high in-degree people show greater behavior drift," and see if the descriptive layer can detect it.

In [9]:
wave1 = edges                              # 第一波 = 已登记的边表
wave2 = ds.load_network(n=25, seed=1)      # 第二波 = 同规模换 seed 重采样

# 造行为向量:入度越高、行为分越高(制造可检出的 influence 代理)
nodes = sorted(set(wave1["source"]) | set(wave1["target"])
               | set(wave2["source"]) | set(wave2["target"]), key=str)
indeg1 = pd.Series(0, index=nodes)
indeg1.update(wave1["target"].value_counts())
rng = np.random.default_rng(7)
behavior1 = np.array([float(indeg1[v]) for v in nodes])
behavior1 = (behavior1 - behavior1.mean()) / (behavior1.std() + 1e-9)
behavior2 = behavior1 + 0.6 * behavior1 + rng.normal(0, 0.3, len(nodes))  # 入度驱动的漂移 + 噪声

st_saom = sv.StudyState()
st_saom.write("sources", "datasets", wave1)
st_saom = sv.tl.saom(st_saom, wave1=wave1, wave2=wave2,
                     behavior1=behavior1, behavior2=behavior2)

saom = st_saom.models["saom"]
coevo = st_saom.diagnostics["coevolution"]
print("方法:", saom["method"], "| 后端:", saom["backend"])
print("近似声明:", coevo["approximation"])

方法: two-wave descriptive (SAOM-style) | 后端: numpy
近似声明: 描述性/简化版:两波间 Jaccard/生成消失/交叉滞后,非 RSiena 基于模拟的矩量法 SAOM 估计


First the network side. Jaccard stability is the overlap ratio of edges between waves; our waves are independently resampled, so it's low (≈0.10) **as expected**—the point is to show the metric itself, not to achieve high stability. Hamming distance is the total difference in adjacency matrices between waves, and generation/dissolution rates decompose it by direction.

In [10]:
print(f"节点数 = {saom['n_nodes']}")
print(f"wave1 边 = {coevo['wave1_ties']} · wave2 边 = {coevo['wave2_ties']}")
print(f"Jaccard 稳定性 = {saom['jaccard']:.4f}")
print(f"Hamming 距离 = {saom['hamming_distance']}")
print(f"生成 / 消失 / 维持 = {saom['ties_created']} / {saom['ties_dropped']} / {saom['ties_maintained']}")
print(f"生成率 = {coevo['creation_rate']:.4f} · 消失率 = {coevo['dissipation_rate']:.4f}")

节点数 = 25
wave1 边 = 101 · wave2 边 = 190
Jaccard 稳定性 = 0.1023
Hamming 距离 = 237
生成 / 消失 / 维持 = 163 / 74 / 27
生成率 = 0.2717 · 消失率 = 0.7327


Now the behavior side. Two cross-lagged proxy measures roughly correspond to the two forces SAOM tries to distinguish: the influence proxy asks "can wave1 in-degree predict behavior change," the selection proxy asks "can wave1 behavior predict degree change." We only embedded the former, so the influence proxy should be positive and clearly larger than the selection proxy.

In [11]:
beh = coevo["behavior"]
print(f"influence 代理 (入度 → 行为变化) = {beh['influence_proxy']:.4f}")
print(f"selection 代理 (行为 → 度变化)   = {beh['selection_proxy']:.4f}")
print(f"行为变化均值 = {beh['behavior_change_mean']:.4f}")

influence 代理 (入度 → 行为变化) = 0.9355
selection 代理 (行为 → 度变化)   = -0.5109
行为变化均值 = -0.1109


The influence proxy is about +0.94, clearly positive—the "in-degree drives behavior drift" we embedded was detected. This is the **descriptive precursor** to SAOM influence/selection decomposition, usable as data exploration before modeling, but it is not structural estimation; don't interpret it as a `RSiena` parameter.

### Plotting SAOM two-wave coevolution

Again, hand-drawn two panels: left decomposes edge fates into persistence/formation/dissolution, right plots wave1 behavior against wave2 behavior, colored by in-degree, making the influence signal "high in-degree nodes show more behavior drift" visible to the eye (points falling above the `y=x` line indicate overall behavior increase).

In [12]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.2))

cats = ["维持", "生成", "消失"]
counts = [saom["ties_maintained"], saom["ties_created"], saom["ties_dropped"]]
axL.bar(cats, counts, color=["#38a169", "#2b6cb0", "#e53e3e"])
axL.set_ylabel("有向连边数")
axL.set_title(f"两波连边命运  (Jaccard={saom['jaccard']:.2f})")
for i, c in enumerate(counts):
    axL.text(i, c, str(c), ha="center", va="bottom")

sc = axR.scatter(behavior1, behavior2, c=behavior1, cmap="viridis",
                 s=45, edgecolor="k", linewidth=0.4)
lims = [min(behavior1.min(), behavior2.min()), max(behavior1.max(), behavior2.max())]
axR.plot(lims, lims, "--", color="grey", lw=1, label="y = x (无变化)")
axR.set_xlabel("wave1 行为 (入度标准化)")
axR.set_ylabel("wave2 行为")
axR.set_title("行为共演化:入度高者行为涨得多")
axR.legend()
fig.colorbar(sc, ax=axR, label="wave1 行为")

fig.tight_layout()
fig.savefig("fig_saom.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("已保存 fig_saom.png")

已保存 fig_saom.png


![SAOM two-wave coevolution](fig_saom.png)

## Stylometry: Recognizing authorship from word usage alone

Switch topics: author attribution. The core intuition is that an author's subject matter changes, but the usage frequency of **function words** like "the/and/of/to" is quite stable, almost unaffected by content—they form a measurable "stylistic fingerprint." Burrows's Delta quantifies this fingerprint: first select the most frequent words in the corpus (MFW), calculate their relative frequencies for each document; then z-score-normalize each word across the entire corpus (removing the effect that "some words are just naturally more common"); the **Delta distance** between two documents is their **mean absolute difference** in these normalized frequencies (equivalent to normalized L1/Manhattan distance); finally, do hierarchical clustering by Delta and attribute each document to the author of "the document closest to it."

The data is 3 authors (austen / dickens / melville) × 3 texts each, each person with a distinct function-word distribution. Known answer: documents by the same author should be mutually closest and cluster together, leave-one-out attribution accuracy should be near 100%. This is the second "known structure recovered" moment of this chapter.

In [13]:
corpus = ds.load_stylometry(seed=0)   # {doc_id: text},9 篇文档
print("语料:", len(corpus), "篇 ·", ", ".join(corpus))
first_id = list(corpus)[0]
print(f"\n示例 [{first_id}] 前 100 字符:\n{corpus[first_id][:100]} ...")

语料: 9 篇 · austen_1, austen_2, austen_3, dickens_1, dickens_2, dickens_3, melville_1, melville_2, melville_3

示例 [austen_1] 前 100 字符:
and a the not and and the the with but as and by the to in a and she to of to with she the of and sh ...


Register the corpus into the `corpus` slot, then run Delta. `n_mfw=20` means using the top 20 most frequent words as features—function words almost necessarily fall in this high-frequency range, so even with just 20 words, the attribution signal is already strong.

In [14]:
st_sty = sv.StudyState()
st_sty.write("corpus", "documents", corpus)
st_sty = sv.tl.stylometry(st_sty, n_mfw=20)

sty = st_sty.models["stylometry"]
print("方法:", sty["method"])
print("MFW 特征数:", sty["n_mfw"], "| 聚类后端:", sty["linkage_backend"])
print(f"留一归属准确率 = {sty['accuracy']:.0%}  ({sty['n_correct']}/{sty['n_documents']} 正确)")

方法: Burrows's Delta (MFW z-score, Manhattan / average linkage)
MFW 特征数: 20 | 聚类后端: scipy
留一归属准确率 = 100%  (9/9 正确)


Attribution per document: each document's true author, its nearest neighbor, predicted author, and Delta distance to nearest neighbor. Two same-author documents are mutual nearest neighbors if attribution is correct.

In [15]:
pd.DataFrame(
    [{"doc": did, "true": r["true_author"], "nearest": r["nearest"],
      "pred": r["predicted_author"], "delta": round(r["delta"], 3),
      "ok": "✓" if r["predicted_author"] == r["true_author"] else "✗"}
     for did, r in sty["attribution"].items()]
)

,doc,true,nearest,pred,delta,ok
0,austen_1,austen,austen_2,austen,0.785,✓
1,austen_2,austen,austen_3,austen,0.741,✓
2,austen_3,austen,austen_2,austen,0.741,✓
3,dickens_1,dickens,dickens_3,dickens,0.843,✓
4,dickens_2,dickens,dickens_3,dickens,0.814,✓
5,dickens_3,dickens,dickens_2,dickens,0.814,✓
6,melville_1,melville,melville_2,melville,0.780,✓
7,melville_2,melville,melville_3,melville,0.695,✓
8,melville_3,melville,melville_2,melville,0.695,✓


All 9 documents attributed correctly, accuracy 100%: each document's nearest neighbor falls under the same author, and cross-author Delta distances are clearly larger than within-author. Function-word fingerprints cleanly separated the three authors.

In [16]:
assert sty["accuracy"] >= 0.6, "同作者应彼此最近,归属准确率应偏高"
print(f"归属准确率 {sty['accuracy']:.0%} ≥ 60%,断言通过。")

归属准确率 100% ≥ 60%,断言通过。


### Dendrogram: same authors fuse at lower heights

Behind attribution is an average-linkage clustering tree. Same-author documents have small Delta, so they fuse **early/low**; different-author clusters have large Delta and merge **late/high**. `sv.pl.dendrogram` reads the scipy-format linkage matrix written by `sv.tl.stylometry` directly to plot, and saves the figure as PNG.

In [17]:
st_sty = sv.pl.dendrogram(st_sty, out="fig_dendro.png")
figrec = st_sty.artifacts["figures"]["dendrogram"]
figpath = figrec["path"] if isinstance(figrec, dict) else figrec
print("树状图已保存:", figpath)

树状图已保存: fig_dendro.png


![Stylometry hierarchical clustering dendrogram](fig_dendro.png)

The three authors form three low-fusion subtrees; the subtrees merge with each other only at high height—the cluster structure perfectly matches the attribution table above.

## Reproducible evidence chain

Each time one of the earlier analysis functions (`ergm` / `saom` / `stylometry` / `dendrogram`) runs successfully, `socialverse` automatically records a step in the evidence chain of the corresponding `StudyState`: which function was used, which slots were read, which slots were written. `st.summary()` unfolds everything at once—which slots are filled, how many steps taken—no need to remember; the registry records as it runs.

In [18]:
print("=== ERGM 主链 ===")
print(st.summary())
print("\n=== SAOM 共演化 ===")
print(st_saom.summary())
print("\n=== 文体计量 ===")
print(st_sty.summary())

=== ERGM 主链 ===
StudyState
  sources: datasets
  models: ergm
  diagnostics: gof
  provenance: 1 step(s)

=== SAOM 共演化 ===
StudyState
  sources: datasets
  models: saom
  diagnostics: coevolution
  provenance: 1 step(s)

=== 文体计量 ===
StudyState
  corpus: documents
  models: stylometry
  artifacts: figures
  provenance: 2 step(s)


Take the stylometry chain as an example, unfold the step-by-step ledger—each step's `requires` (which slots were read) and `produces` (which slots were written) are at a glance, this is the underlying account that makes analysis traceable and reproducible.

In [19]:
for rec in st_sty.provenance:
    print(f"step {rec['step']}: {rec['function']}")
    print(f"    requires: {rec['requires']}")
    print(f"    produces: {rec['produces']}")

step 1: socialverse.tl._stylometry.stylometry
    requires: {'corpus': ['documents']}
    produces: {'models': ['stylometry'], 'artifacts': ['figures']}
step 2: socialverse.pl._figure2.dendrogram
    requires: {'models': ['stylometry']}
    produces: {'artifacts': ['figures']}


## Summary

We went through two pipelines in the same tutorial: ERGM + SAOM extracted **reciprocity, transitive closure, two-wave evolution** from a social network, benchmarking R's `ergm`/`statnet` and `RSiena`; Burrows's Delta cleanly attributed 9 documents by 3 authors using function words alone, benchmarking R's `stylo`. These three methods are either natively absent or scattered across Python's ecosystem; here we collect them into one unified workflow, and all pass the test "can known answers be recovered."

Compared to scattered scripts, `socialverse` adds two things: first, **honest labeling of "exact vs. approximation"**—ERGM approximates MCMC-MLE with MPLE, SAOM only does descriptive layer rather than true SIENA estimation, and where approximations fail (like the huge gap in GoF's triangle terms) is laid bare; second, an **evidence chain built as we run**, making each step's source and effect traceable. The next tutorial [17_text_scaling](17_text_scaling.ipynb) continues the text thread, turning toward estimating latent ideological positions from text.